# Manager World Cup History

## Objective

Create pre-tournament manager experience features for each World Cup team. The key modeling concern is avoiding forward leakage: manager records for a target World Cup should only use matches before that tournament.

## Inputs

- `worldcup::manager_appearances`
- `1.DataCleaning-R/Data/RDS/CoachTeamMatchesBeforeWC.rds`

## Outputs

- `1.DataCleaning-R/Data/RDS/ManagerWC_history.rds`
- `1.DataCleaning-R/Data/CSV/ManagerWC_history.csv`

## Packages


In [1]:
library(worldcup)
library(tidyverse)
library(here)

Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'purrr' was built under R version 4.4.3"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.0     v stringr   1.6.0
v ggplot2   4.0.1     v tibble    3.2.1
v lubridate 1.9.4     v tidyr     1.3.1
v purrr     1.2.1     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
here() starts at /Users/eialnisman/Desktop/WC2026Forecast



## Source Manager Games

Load historical manager appearances from the World Cup package and inspect the available manager-game structure.


In [2]:
manager_games<-worldcup::manager_appearances


manager_games %>%
        group_by(manager_id) %>%
            summarize(games=sum(n()), groups = sum(stage_name == "group_stage"), r16 = sum(stage_name == "Round of 16"), qf = (sum(stage_name == "quarter_finals")), sf = sum(stage_name == "semi_finals"), p3 =sum(stage_name== "third-place match"), finals = sum(stage_name=="final"), wcs_coached = n_distinct(tournament_id)) %>%
                print(n=10)

# A tibble: 475 x 9
   manager_id games groups   r16    qf    sf    p3 finals wcs_coached
   <chr>      <int>  <int> <int> <int> <int> <int>  <int>       <int>
 1 M-001          4      0     0     0     0     0      0           1
 2 M-002          3      0     0     0     0     0      0           1
 3 M-003          3      0     0     0     0     0      0           1
 4 M-004          8      0     0     0     0     0      0           2
 5 M-005          3      0     0     0     0     0      0           1
 6 M-006          8      0     0     0     0     0      0           2
 7 M-007          3      0     0     0     0     0      0           1
 8 M-008          3      0     0     0     0     0      0           1
 9 M-009          3      0     0     0     0     0      0           1
10 M-010          6      0     0     0     0     0      0           2
# i 465 more rows


## Build Pre-Tournament History

Summarize each manager only using tournaments that occurred before the target tournament.


In [3]:
target_tournaments <- c("WC-2010", "WC-2014", "WC-2018", "WC-2022", "WC-2026")

summarize_manager_history <- function(target_tournament) {
    manager_games %>%
        filter(tournament_id < target_tournament) %>%
        group_by(manager_id) %>%
        summarize(
            prior_wc_matches_coached = n(),
            prior_wc_group_matches_coached = sum(stage_name == "group stage"),
            prior_wc_round_of_16_matches_coached = sum(stage_name == "round of 16"),
            prior_wc_quarterfinal_matches_coached = sum(stage_name %in% c("quarter-final", "quarter-finals")),
            prior_wc_semifinal_matches_coached = sum(stage_name %in% c("semi-final", "semi-finals")),
            prior_wc_third_place_matches_coached = sum(stage_name == "third-place match"),
            prior_wc_final_matches_coached = sum(stage_name == "final"),
            prior_world_cups_coached = n_distinct(tournament_id),
            .groups = "drop"
        ) %>%
        mutate(tournament_id = target_tournament)
}

before2010 <- summarize_manager_history("WC-2010")
before2014 <- summarize_manager_history("WC-2014")
before2018 <- summarize_manager_history("WC-2018")
before2022 <- summarize_manager_history("WC-2022")
before2026 <- summarize_manager_history("WC-2026")

## Combine Tournament Windows

Bind the per-tournament summaries into one table with an explicit target tournament column.


In [4]:
manager_history <- bind_rows(
  before2010,
  before2014,
  before2018,
  before2022,
  before2026
)

## Match Current Team Managers

Join manager history onto the manager/team/tournament matching table used for 2026-style features.


In [5]:
coach_team_matches <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "CoachTeamMatchesBeforeWC.rds"))

matchingmanagers <- coach_team_matches %>%
    select(
        tournament_id,
        team_id,
        team_name,
        team_code,
        manager_id,
        coach_name_nft,
        coach_url_nft,
        pre_wc_team_matches_coached = coach_team_matches_before_wc,
        coach_team_matches_source,
        coach_team_matches_cutoff_date
    )
    
matchingmanagers %>%
    print(n=10)

# A tibble: 176 x 10
   tournament_id team_id team_name team_code manager_id coach_name_nft       
   <chr>         <chr>   <chr>     <chr>     <chr>      <chr>                
 1 WC-2010       T-01    Algeria   DZA       M-366      "Sa\u00e2dane, Rabah"
 2 WC-2010       T-03    Argentina ARG       M-241      "Maradona, Diego"    
 3 WC-2010       T-04    Australia AUS       M-448      "Verbeek, Pim"       
 4 WC-2010       T-09    Brazil    BRA       M-108      "Dunga,"             
 5 WC-2010       T-11    Cameroon  CMR       M-218      "Le Guen, Paul"      
 6 WC-2010       T-13    Chile     CHL       M-043      "Bielsa, Marcelo"    
 7 WC-2010       T-22    Denmark   DNK       M-299      "Olsen, Morten"      
 8 WC-2010       T-28    England   ENG       M-071      "Capello, Fabio"     
 9 WC-2010       T-30    France    FRA       M-105      "Domenech, Raymond"  
10 WC-2010       T-31    Germany   DEU       M-234      "L\u00f6w, Joachim"  
# i 166 more rows
# i 4 more variables: coa

## Fill Missing History

Managers without prior World Cup history are assigned zero prior experience rather than being dropped.


In [6]:
manager_history <- matchingmanagers %>%
    left_join(manager_history, by=c("manager_id", "tournament_id")) %>%
    mutate(across(c(prior_wc_matches_coached, prior_wc_group_matches_coached, prior_wc_round_of_16_matches_coached, prior_wc_quarterfinal_matches_coached, prior_wc_semifinal_matches_coached, prior_wc_third_place_matches_coached, prior_wc_final_matches_coached, prior_world_cups_coached), ~replace_na(.x, 0L)))

manager_history %>%
    print(n=10)

# A tibble: 176 x 18
   tournament_id team_id team_name team_code manager_id coach_name_nft       
   <chr>         <chr>   <chr>     <chr>     <chr>      <chr>                
 1 WC-2010       T-01    Algeria   DZA       M-366      "Sa\u00e2dane, Rabah"
 2 WC-2010       T-03    Argentina ARG       M-241      "Maradona, Diego"    
 3 WC-2010       T-04    Australia AUS       M-448      "Verbeek, Pim"       
 4 WC-2010       T-09    Brazil    BRA       M-108      "Dunga,"             
 5 WC-2010       T-11    Cameroon  CMR       M-218      "Le Guen, Paul"      
 6 WC-2010       T-13    Chile     CHL       M-043      "Bielsa, Marcelo"    
 7 WC-2010       T-22    Denmark   DNK       M-299      "Olsen, Morten"      
 8 WC-2010       T-28    England   ENG       M-071      "Capello, Fabio"     
 9 WC-2010       T-30    France    FRA       M-105      "Domenech, Raymond"  
10 WC-2010       T-31    Germany   DEU       M-234      "L\u00f6w, Joachim"  
# i 166 more rows
# i 12 more variables: co

## Save

Persist both RDS and CSV versions for later modeling and inspection.


In [7]:
saveRDS(manager_history, here("1.DataCleaning-R", "Data", "RDS", "ManagerWC_history.rds"))
write_csv(manager_history, here("1.DataCleaning-R", "Data", "CSV", "ManagerWC_history.csv"))